In [1]:
import os
import copy
import time
import json
import random
import shutil

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    balanced_accuracy_score,
)

# ================= CONFIGURATION & HYPERPARAMÈTRES =================

# Mise à jour du chemin racine vers votre nouveau projet
BASE_PATH = "get_project_root()"

# Chemin direct vers le dataset traité (contient train/ et val/)
# Note : Assurez-vous que les sous-dossiers sont exactement : "male", "femelle", "indeterminee", "couple"
DATA_DIR = os.path.join(BASE_PATH, "data/02_processed/n_mobilenet_mfic_dataset")

# Nom de l'expérience pour les logs (output)
RUN_NAME = "mobilenet_v3_mfic_4classes_v2"
OUTPUT_DIR = os.path.join(BASE_PATH, "outputs", RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paramètres d'entraînement
BATCH_SIZE = 8          # Faible batch size car dataset potentiellement petit ou GPU limité
NUM_EPOCHS = 150        # Nombre d'époques (arrêt possible via Ctrl+C, le meilleur modèle est sauvegardé)
INPUT_SIZE = 224        # Taille standard pour MobileNet/ResNet
NUM_WORKERS = 4         # Parallélisation du chargement des données
SEED = 42               # Grain pour la reproductibilité

# Fine-tuning (Stratégie pragmatique)
# On ne réentraîne que les 3 derniers blocs convolutionnels pour ne pas détruire les features bas niveau.
UNFREEZE_LAST_N_FEATURE_BLOCKS = 3
LR_HEAD = 5e-4          # Learning rate plus élevé pour la nouvelle tête de classification
LR_BACKBONE = 1e-4      # Learning rate plus faible pour le backbone (extraction de features)
WEIGHT_DECAY = 1e-4     # Régularisation L2

# Gestion du déséquilibre (Crucial pour les classes rares comme "indeterminee" ou "couple")
USE_WEIGHTED_SAMPLER = True  # Recommande : True. Force le réseau à voir autant de chaque classe par époque.
USE_WEIGHTED_CE = False      # Alternative si le sampler est False (pondération dans la Loss function).

# Règle de décision à seuils (Logique métier)
# Plutôt que de prendre le max brut (Argmax), on privilégie certaines classes si leur probabilité dépasse un seuil.
# Hiérarchie de décision : Indeterminee > Couple > Femelle > Male (classe par défaut souvent majoritaire)
THRESHOLD_FEMALE = 0.40
THRESHOLD_INDET = 0.50
THRESHOLD_COUPLE = 0.50

# Si True, ignore les seuils ci-dessus et prend juste la probabilité maximale (moins risqué mais moins fin).
USE_ARGMAX_DECISION = False

# Régularisation (Label Smoothing)
# Empêche le modèle d'être trop sûr de lui (évite les probas à 1.0 qui causent de l'overfitting).
USE_LABEL_SMOOTHING = True
LABEL_SMOOTHING = 0.05

# ================= OUTILS & UTILITAIRES =================

def set_seed(seed: int = 42):
    """Fixe les grains aléatoires pour assurer la reproductibilité des résultats."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True # Optimisation vitesse sur GPU

def plot_history(history, save_path):
    """Génère les courbes de Loss et d'Accuracy après l'entraînement."""
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))

    # Courbe de perte (Loss)
    ax[0].plot(history["train_loss"], label="Train Loss")
    ax[0].plot(history["val_loss"], label="Val Loss")
    ax[0].set_title("Evolution de la Loss")
    ax[0].set_xlabel("Epochs")
    ax[0].set_ylabel("Loss")
    ax[0].legend()
    ax[0].grid(True)

    # Courbe des métriques
    ax[1].plot(history["train_acc"], label="Train Acc (argmax)")
    ax[1].plot(history["val_acc"], label="Val Acc (decision)")
    ax[1].plot(history["val_bal_acc"], label="Val Balanced Acc")
    ax[1].plot(history["val_f1_female"], label="Val F1 (femelle)")
    ax[1].set_title("Evolution des Métriques")
    ax[1].set_xlabel("Epochs")
    ax[1].set_ylabel("Score")
    ax[1].legend()
    ax[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "results.png"))
    plt.close()

def plot_confusion_matrix(y_true, y_pred, classes, save_path):
    """Affiche et sauvegarde la matrice de confusion (brute et normalisée)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    # Normalisation pour voir les pourcentages d'erreur par classe réelle
    cm_norm = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1.0)

    # 1. Matrice brute
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.ylabel("Vraie Classe")
    plt.xlabel("Classe Prédite")
    plt.title("Matrice de Confusion (Quantités)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "confusion_matrix.png"))
    plt.close()

    # 2. Matrice normalisée
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0.0, vmax=1.0, xticklabels=classes, yticklabels=classes)
    plt.ylabel("Vraie Classe")
    plt.xlabel("Classe Prédite")
    plt.title("Matrice de Confusion (Normalisée)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "confusion_matrix_normalized.png"))
    plt.close()

def build_weighted_sampler(image_folder_dataset):
    """
    Crée un sampler qui tire plus souvent les images des classes rares.
    Essentiel pour éviter que le modèle ne prédise toujours 'male' si c'est la classe majoritaire.
    """
    targets = [y for _, y in image_folder_dataset.samples]
    class_count = np.bincount(targets)
    class_weight = 1.0 / np.maximum(class_count, 1) # Poids inverse de la fréquence
    
    sample_weights = np.array([class_weight[t] for t in targets], dtype=np.float64)
    sample_weights = torch.from_numpy(sample_weights)

    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )
    return sampler, class_count

class ImageFolderWithPaths(datasets.ImageFolder):
    """
    Surcharge de ImageFolder pour retourner aussi le chemin du fichier.
    Utile pour identifier quelles images précises posent problème (debugging visuel).
    """
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path, _ = self.samples[index]
        return img, label, path

def predict_with_thresholds_4(probs, idx_f, idx_m, idx_i, idx_c, thr_f, thr_i, thr_c, labels_like):
    """
    Logique de décision métier post-traitement.
    Si la proba de 'indeterminee' > 0.5, on classe indet (priorité sécurité).
    Idem pour couple et femelle. Sinon par défaut : Male.
    """
    p_f = probs[:, idx_f]
    p_i = probs[:, idx_i]
    p_c = probs[:, idx_c]

    pred = torch.full_like(labels_like, fill_value=idx_m)  # Défaut : MALE
    pred = torch.where(p_f >= thr_f, torch.full_like(pred, idx_f), pred)
    pred = torch.where(p_c >= thr_c, torch.full_like(pred, idx_c), pred)
    pred = torch.where(p_i >= thr_i, torch.full_like(pred, idx_i), pred) # Priorité max (écrasera les précédents si conflit)
    return pred

# ================= FONCTION PRINCIPALE (MAIN) =================

def main():
    set_seed(SEED)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Utilisation du device : {device}")
    print(f"[INFO] Dossier de données : {DATA_DIR}")

    # 1) Augmentation de données (Data Augmentation)
    # Transformations pour rendre le modèle robuste aux variations de lumière et d'orientation
    data_transforms = {
        "train": transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5), # Symétrie horizontale
            transforms.RandomRotation(15),          # Rotation légère
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), # Translation légère
            transforms.ColorJitter(brightness=0.1, contrast=0.1),       # Variation couleur
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        "val": transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
    }

    # 2) Chargement des Datasets
    image_datasets = {
        x: ImageFolderWithPaths(os.path.join(DATA_DIR, x), data_transforms[x])
        for x in ["train", "val"]
    }

    class_names = image_datasets["train"].classes
    num_classes = len(class_names)
    dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}

    print(f"[INFO] Classes détectées : {class_names}")
    print(f"[INFO] Train size : {dataset_sizes['train']} | Val size : {dataset_sizes['val']}")

    # Récupération des indices (Mapping Classe -> Entier)
    class_to_idx = image_datasets["train"].class_to_idx
    IDX_F = class_to_idx.get("femelle", None)
    IDX_M = class_to_idx.get("male", None)
    IDX_I = class_to_idx.get("indeterminee", None)
    IDX_C = class_to_idx.get("couple", None)

    # Vérification de sécurité pour être sûr que les 4 classes existent
    if any(v is None for v in [IDX_F, IDX_M, IDX_I, IDX_C]):
        raise ValueError(
            f"ERREUR CRITIQUE : Classes manquantes ou mal nommées.\n"
            f"Attendu : ['femelle', 'male', 'indeterminee', 'couple']\n"
            f"Trouvé : {list(class_to_idx.keys())}"
        )

    # 3) Préparation du Sampler (Pondération des données)
    train_sampler = None
    if USE_WEIGHTED_SAMPLER:
        train_sampler, class_count = build_weighted_sampler(image_datasets["train"])
        print(f"[INFO] WeightedSampler ACTIF. Répartition brute train : {class_count}")
    else:
        print("[INFO] WeightedSampler INACTIF. Entraînement standard.")

    # 4) Création des DataLoaders
    dataloaders = {
        "train": torch.utils.data.DataLoader(
            image_datasets["train"],
            batch_size=BATCH_SIZE,
            shuffle=(train_sampler is None), # Shuffle seulement si pas de sampler
            sampler=train_sampler,
            num_workers=NUM_WORKERS,
            drop_last=False,
            pin_memory=True,
        ),
        "val": torch.utils.data.DataLoader(
            image_datasets["val"],
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            drop_last=False,
            pin_memory=True,
        ),
    }

    # 5) Architecture du Modèle (MobileNetV3 Large)
    print("[INFO] Chargement de MobileNetV3 Large...")
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    
    # On gèle tous les poids par défaut (Transfer Learning)
    for p in model.parameters():
        p.requires_grad = False
    
    # On débloque les derniers blocs pour permettre au modèle d'apprendre des features spécifiques aux gammares
    if UNFREEZE_LAST_N_FEATURE_BLOCKS > 0:
        for p in model.features[-UNFREEZE_LAST_N_FEATURE_BLOCKS:].parameters():
            p.requires_grad = True
    
    # Récupération dynamique de la taille de sortie du backbone
    # (Astuce technique pour éviter de hardcoder "960")
    with torch.no_grad():
        dummy_input = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
        features = model.features(dummy_input)
        features = model.avgpool(features)
        backbone_out = torch.flatten(features, 1).shape[1]
    
    # Remplacement de la "Tête" (Classifier)
    # Ajout de LayerNorm et Dropout pour stabiliser l'apprentissage
    model.classifier = nn.Sequential(
        nn.Linear(backbone_out, 512),
        nn.LayerNorm(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5), # Réduit l'overfitting
        nn.Linear(512, num_classes),
    )
    
    model = model.to(device)

    # 6) Fonction de Coût (Loss)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # 7) Optimiseur
    # On sépare les paramètres pour appliquer des Learning Rates différents (Backbone vs Head)
    head_params = list(model.classifier.parameters())
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n]

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params, "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    # Scheduler : réduit le LR si le score ne s'améliore plus (plateau)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8)

    # 8) Boucle d'entraînement
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_score = -1.0 # On cherchera à maximiser le F1-Score

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_bal_acc": [], "val_f1_female": []}

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print("-" * 40)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            seen = 0
            
            # Listes pour calcul des métriques globales en fin d'époque
            y_true_epoch = []
            y_pred_epoch = []

            for inputs, labels, _ in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                # Statistiques
                bs = inputs.size(0)
                seen += bs
                running_loss += loss.item() * bs

                # Prédictions
                if phase == "train":
                    # En train, on regarde juste l'argmax pour la vitesse
                    _, preds = torch.max(outputs, 1)
                else:
                    # En val, on applique notre logique de seuil complexe
                    probs = torch.softmax(outputs, dim=1)
                    if USE_ARGMAX_DECISION:
                        _, preds = torch.max(outputs, 1)
                    else:
                        preds = predict_with_thresholds_4(
                            probs, IDX_F, IDX_M, IDX_I, IDX_C,
                            THRESHOLD_FEMALE, THRESHOLD_INDET, THRESHOLD_COUPLE,
                            labels
                        )

                running_corrects += torch.sum(preds == labels.data).item()
                y_true_epoch.extend(labels.cpu().tolist())
                y_pred_epoch.extend(preds.cpu().tolist())

            epoch_loss = running_loss / max(seen, 1)
            epoch_acc = running_corrects / max(seen, 1)

            if phase == "train":
                history["train_loss"].append(epoch_loss)
                history["train_acc"].append(epoch_acc)
                print(f"Train | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
            else:
                # Calcul des métriques avancées
                val_bal_acc = balanced_accuracy_score(y_true_epoch, y_pred_epoch)
                val_f1_female = f1_score(np.array(y_true_epoch) == IDX_F, np.array(y_pred_epoch) == IDX_F, zero_division=0)

                history["val_loss"].append(epoch_loss)
                history["val_acc"].append(epoch_acc)
                history["val_bal_acc"].append(val_bal_acc)
                history["val_f1_female"].append(val_f1_female)

                print(f"Val   | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Bal_Acc: {val_bal_acc:.4f} F1_Fem: {val_f1_female:.4f}")

                # Le scheduler se base sur le F1 score femelle (critique)
                scheduler.step(val_f1_female)

                # Sauvegarde du meilleur modèle
                if val_f1_female > best_score:
                    best_score = val_f1_female
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(best_model_wts, os.path.join(OUTPUT_DIR, "best_model.pth"))
                    print(">>> Meilleur modèle sauvegardé (basé sur F1 Femelle)")

    time_elapsed = time.time() - since
    print(f"\nEntraînement terminé en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Meilleur F1 Femelle : {best_score:.4f}")

    # 9) Évaluation Finale et Analyse des Erreurs
    print("\n--- ANALYSE DES ERREURS SUR LA VALIDATION ---")
    model.load_state_dict(best_model_wts)
    model.eval()

    y_true = []
    y_pred = []
    y_probs = []
    val_paths = []

    with torch.no_grad():
        for inputs, labels, paths in dataloaders["val"]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            
            if USE_ARGMAX_DECISION:
                _, preds = torch.max(outputs, 1)
            else:
                preds = predict_with_thresholds_4(
                    probs, IDX_F, IDX_M, IDX_I, IDX_C,
                    THRESHOLD_FEMALE, THRESHOLD_INDET, THRESHOLD_COUPLE,
                    labels
                )
            
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            y_probs.extend(probs.cpu().tolist())
            val_paths.extend(paths)

    # Génération des rapports
    plot_confusion_matrix(y_true, y_pred, class_names, OUTPUT_DIR)
    plot_history(history, OUTPUT_DIR)
    
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
        f.write(report)
    print(report)

    # Export des images mal classées pour analyse visuelle
    errors_dir = os.path.join(OUTPUT_DIR, "errors_val_visualisation")
    if os.path.exists(errors_dir): shutil.rmtree(errors_dir)
    os.makedirs(errors_dir)

    y_probs = np.array(y_probs)
    errors_indices = np.where(np.array(y_true) != np.array(y_pred))[0]

    print(f"[INFO] Sauvegarde des {len(errors_indices)} erreurs dans {errors_dir}...")
    
    for i in errors_indices:
        src = val_paths[i]
        true_cls = class_names[y_true[i]]
        pred_cls = class_names[y_pred[i]]
        
        # On renomme l'image avec les probabilités pour comprendre pourquoi le modèle s'est trompé
        # Format: VRAI_vs_PRED__confiance.jpg
        conf = y_probs[i][y_pred[i]] * 100
        fname = f"VRAI-{true_cls}_PRED-{pred_cls}_conf{conf:.1f}.jpg"
        
        shutil.copy2(src, os.path.join(errors_dir, fname))

    print(f"Terminé. Résultats disponibles dans : {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

[INFO] Utilisation du device : cuda:0
[INFO] Dossier de données : get_project_root()data/02_processed/n_mobilenet_mfic_dataset
[INFO] Classes détectées : ['couple', 'femelle', 'indeterminee', 'male']
[INFO] Train size : 413 | Val size : 75
[INFO] WeightedSampler ACTIF. Répartition brute train : [ 17 152  89 155]
[INFO] Chargement de MobileNetV3 Large...

Epoch 1/150
----------------------------------------
Train | Loss: 1.1882 Acc: 0.4673
Val   | Loss: 0.9925 Acc: 0.6133 Bal_Acc: 0.7066 F1_Fem: 0.6275
>>> Meilleur modèle sauvegardé (basé sur F1 Femelle)

Epoch 2/150
----------------------------------------
Train | Loss: 0.8642 Acc: 0.6659
Val   | Loss: 1.0200 Acc: 0.5867 Bal_Acc: 0.6870 F1_Fem: 0.6275

Epoch 3/150
----------------------------------------
Train | Loss: 0.6890 Acc: 0.7554
Val   | Loss: 0.8470 Acc: 0.7200 Bal_Acc: 0.7916 F1_Fem: 0.7600
>>> Meilleur modèle sauvegardé (basé sur F1 Femelle)

Epoch 4/150
----------------------------------------
Train | Loss: 0.6778 Acc: 0.772